In [1]:
import gradio as gr
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image

# 1. Load your saved multi-head model
model = tf.keras.models.load_model('/kaggle/input/models/haricharanpampati/task1/keras/default/1/model.keras') #[cite: 2]

# Gender dictionary mapping based on your notebook
gender_dict = {0: 'Male', 1: 'Female'}

def detect_hair_length(image_pil):
    """
    OpenCV heuristic to estimate hair length.
    Detects the face, isolates the region below the face (shoulders), 
    and measures the edge density to classify 'Long' vs 'Short' hair.
    """
    # CRITICAL FIX: Convert to RGB safely to avoid RGBA shape issues with cv2
    img_rgb = np.array(image_pil.convert('RGB'))
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    
    # Load the default Haar Cascade for face detection
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    
    # If no face is found, default to Short
    if len(faces) == 0:
        return "Short"
        
    x, y, w, h = faces[0]
    
    # Define a Region of Interest (ROI) below the face
    img_height, img_width = gray.shape
    roi_top = min(y + h, img_height)
    roi_bottom = min(y + h + int(h * 0.8), img_height)
    roi_left = max(x - int(w * 0.2), 0)
    roi_right = min(x + w + int(w * 0.2), img_width)
    
    if roi_top >= roi_bottom:
        return "Short"
        
    shoulder_roi = gray[roi_top:roi_bottom, roi_left:roi_right]
    
    # Use Canny edge detection to find texture/strands (hair)
    edges = cv2.Canny(shoulder_roi, 50, 150)
    edge_density = np.sum(edges / 255.0) / (shoulder_roi.shape[0] * shoulder_roi.shape[1] + 1e-5)
    
    # Threshold for edge density to classify as long hair
    if edge_density > 0.04:
        return "Long"
    else:
        return "Short"

def task1_pipeline(input_image):
    try:
        # --- Step A: Base Model Prediction ---
        # Preprocess the image to match the model's expected input shape (128, 128, 1)
        img_gray = input_image.convert('L')
        # Use Image.Resampling.BILINEAR for newer PIL versions
        img_resized = img_gray.resize((128, 128), Image.Resampling.BILINEAR if hasattr(Image, 'Resampling') else Image.BILINEAR)
        img_array = np.array(img_resized).reshape(1, 128, 128, 1) / 255.0
        
        predictions = model.predict(img_array)
        
        # Safely extract floats from the prediction arrays
        pred_gender_prob = float(predictions[0][0][0])
        pred_age_norm = float(predictions[1][0][0])
        
        # Extract un-normalized age and mapped gender
        predicted_age = int(pred_age_norm * 100)
        base_gender = gender_dict[int(np.round(pred_gender_prob))]
        
        # --- Step B: Hair Length Detection ---
        hair_length = detect_hair_length(input_image)
        
        # --- Step C: Task 1 Override Logic ---
        final_gender = base_gender
        
        # Apply logic exclusively for individuals aged between 20 and 30[cite: 5]
        if 20 <= predicted_age <= 30:
            if hair_length == "Long":
                final_gender = "Female" # Detect as female even if male[cite: 5]
            else:
                final_gender = "Male" # Detect as male even if female[cite: 5]
                
        # If outside 20-30, final_gender remains the base_gender regardless of hair length[cite: 5]

        return str(predicted_age), base_gender, hair_length, final_gender
        
    except Exception as e:
        # If an error occurs, print the Python exception to the GUI instead of failing silently[cite: 3]
        error_msg = f"Exception: {str(e)}"
        return error_msg, error_msg, error_msg, error_msg

# Build the Graphical User Interface (GUI)[cite: 5]
gui = gr.Interface(
    fn=task1_pipeline,
    inputs=gr.Image(type="pil", label="Upload Person Image"),
    outputs=[
        gr.Textbox(label="Predicted Age"),
        gr.Textbox(label="Base Model Gender"),
        gr.Textbox(label="Detected Hair Length"),
        gr.Textbox(label="Final Task 1 Gender (Adjusted)")
    ],
    title="Task 1: Long Hair Identification",
    description="Upload an image to predict age and gender. If the user is aged 20-30, gender prediction is overridden based on hair length." #[cite: 5]
)

# Launch the GUI inside the notebook
gui.launch(share=True)

2026-10-08 15:38:41.876090: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b224fd871522dd596a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 424ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Created dataset file at: .gradio/flagged/dataset1.csv
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
